# FLUKE Sentiment Analysis with GPT-5

This notebook evaluates sentiment analysis robustness using OpenAI's GPT-5 model with FLUKE linguistic modifications.

In [34]:
# Standard imports
from datasets import load_dataset
import dspy
import openai
import os
import pandas as pd
import json
import glob
import time
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models
)

In [35]:
# Load environment variables
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## GPT-5 Configuration

In [46]:
# Select GPT-5 configuration
CONFIG_NAME = 'standard'  # Options: 'standard', 'detailed', 'turbo'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with GPT-5
# GPT-5 only supports temperature=1
lm = dspy.LM(MODEL_ID, max_tokens=20_000, temperature=1)
dspy.configure(lm=lm)

Configuration: standard
Model: gpt-5 (openai/gpt-5)
Description: Standard reasoning approach with GPT-5


## Load Data

In [47]:
# Load sentiment dataset from JSON file
with open('../../../data/train_dev_test_data/sent/test.json', 'r') as f:
    ds = json.load(f)
print(f"Dataset size: {len(ds)}")

# Create examples
examples = [
    dspy.Example({
        "text": remove_space(item["sentence"]),
        "label": item["label"]
    }).with_inputs("text")
    for item in ds
]

# Test example
example = examples[0]
print(f"\nExample text: {example.text}")
print(f"Label: {example.label}")

Dataset size: 872

Example text: it's a charming and often affecting journey.
Label: 1


## Define Task with GPT-5

In [48]:
class GPT5Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Analyze the emotional tone, word choice, and overall sentiment. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class GPT5SentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(GPT5Sentiment)

    def forward(self, text):
        return self.prog(text=text)

# Initialize module
gpt5_sentiment = GPT5SentimentModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    try:
        pred = prediction.label
        # Handle both string and parsed outputs
        if isinstance(pred, str):
            parsed_answer = extract_classification_prediction(pred)
        else:
            # If it's already parsed, extract the value
            parsed_answer = str(pred)
        return parsed_answer == str(true.label)
    except Exception as e:
        print(f"Error parsing prediction: {e}")
        print(f"Prediction object: {prediction}")
        print(f"Prediction type: {type(prediction)}")
        return False

In [49]:
# Test single example
pred = gpt5_sentiment(text=example.text)
print(f"Text: {example.text}")
print(f"True Label: {example.label}")
print(f"Prediction: {pred.label}")
print(f"Correct: {eval_metric(example, pred)}")

Text: it's a charming and often affecting journey.
True Label: 1
Prediction: 1
Correct: True


## Evaluate Original Dataset

In [51]:
# GPT-5 can handle larger batches than o3
# TEST_SIZE = 200  # Can increase for GPT-5
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with GPT-5...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=4,  # GPT-5 can handle more threads
    display_progress=True,
    # display_table=10,
    return_outputs=True,
    return_all_scores=True,
    provide_traceback=True
)

results = evaluate(gpt5_sentiment)

# Save results
items = []
for sample in results[1]:
    items.append({
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    })

df_result = pd.DataFrame(items)
output_file = f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
df_result.to_csv(output_file, index=False)

print(f"\nGPT-5 Accuracy: {results[0]:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 872 examples with GPT-5...
Average Metric: 829.00 / 872 (95.1%): 100%|██████████| 872/872 [00:03<00:00, 248.07it/s] 

2025/08/15 14:12:02 INFO dspy.evaluate.evaluate: Average Metric: 829 / 872 (95.1%)




GPT-5 Accuracy: 95.070
Results saved to: ../results/sa/gpt-5-standard-0shot-sst2.csv


## Evaluate Modifications

In [52]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with GPT-5."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r['modified_text']),
            "original_text": remove_space(r['original_text']),
            "label": int(r.get('modified_label', r['label'])),
            "original_label": int(r['label'])
        }).with_inputs("text")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=4,  # GPT-5 can handle more threads
        display_progress=True,
        display_table=1,
        return_outputs=True,
        return_all_scores=True
    )
    
    return evaluate(program)

In [55]:
# Load original predictions
original_pred_file = f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original GPT-5 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test all modifications with GPT-5
json_files = glob.glob('../../../data/modified_data/sa/*_100.json')
# GPT-5 can handle more modifications
# test_modifications = [
#     'typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json',
#     'negation_100.json', 'sentiment_100.json', 'active_to_passive_100.json'
# ]
# json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"\nTesting {len(json_files)} modifications with GPT-5...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # GPT-5 can handle larger samples
    results_mod = evaluate_modified_set(data, gpt5_sentiment, max_samples=150)
    
    # Process results
    items = []
    for sample in results_mod[1]:
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label']),
            'raw_output': sample[1]['label']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds['text'] == item['original_text']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod[0]:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(2)  # Shorter delay for GPT-5

Loaded original GPT-5 predictions from ../results/sa/gpt-5-standard-0shot-sst2.csv

Testing 17 modifications with GPT-5...

Processing: casual_100.json
Average Metric: 94.00 / 100 (94.0%): 100%|██████████| 100/100 [01:32<00:00,  1.08it/s]

2025/08/15 14:24:19 INFO dspy.evaluate.evaluate: Average Metric: 94 / 100 (94.0%)


,text,original_text,example_label,original_label,pred_label,eval_metric
0,it shows that fincher is a director who skillfully uses tech skill...,it confirms fincher's status as a film maker who artfully bends te...,1,1,1,✔️ [True]


Accuracy: 94.000
Saved to: ../results/sa/gpt-5-standard-0shot-casual_100.csv

Processing: discourse_100.json
Average Metric: 88.00 / 99 (88.9%): 100%|██████████| 99/99 [01:24<00:00,  1.17it/s]

2025/08/15 14:25:46 INFO dspy.evaluate.evaluate: Average Metric: 88 / 99 (88.9%)


...

,text,original_text,example_label,original_label,pred_label,eval_metric
0,"As the two leads, Lathan and Diggs are seen as charming and are pe...","as the two leads, lathan and diggs are charming and have chemistry...",1,1,1,✔️ [True]


Accuracy: 93.000
Saved to: ../results/sa/gpt-5-standard-0shot-active_to_passive_100.csv

Processing: negation_100.json
Average Metric: 82.00 / 96 (85.4%): 100%|██████████| 96/96 [00:00<00:00, 1708.63it/s]

2025/08/15 14:38:49 INFO dspy.evaluate.evaluate: Average Metric: 82 / 96 (85.4%)


,text,original_text,example_label,original_label,pred_label,eval_metric
0,What seldom distinguishes Time of Favor from countless other thril...,what distinguishes time of favor from countless other thrillers is...,1,1,1,✔️ [True]


Accuracy: 85.420
Saved to: ../results/sa/gpt-5-standard-0shot-negation_100.csv


In [56]:
# Aggregate all modification results
result_files = glob.glob(f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='sentiment_analysis',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")


gpt-5-standard Results Summary:
                    modification  original_res  modified_res  difference  \
0              temporal_bias_100         0.940         0.970       0.030   
1          geographical_bias_100         0.920         0.930       0.010   
2                length_bias_100         0.970         0.940      -0.030   
3                  typo_bias_100         0.950         0.960       0.010   
4             capitalization_100         0.960         0.960      -0.000   
5                punctuation_100         0.910         0.940       0.030   
6                 derivation_100         0.954         0.943      -0.011   
7              compound_word_100         0.968         0.958      -0.011   
8          active_to_passive_100         0.950         0.930      -0.020   
9           grammatical_role_100         0.955         0.955      -0.000   
10  coordinating_conjunction_100         0.950         0.960       0.010   
11       concept_replacement_100         0.950         

,task,model,modification,original_res,modified_res,difference,pct_difference,p_value,samples
0,sentiment_analysis,gpt-5-standard,temporal_bias_100,0.940000,0.970000,0.030000,3.190000,0.309000,100
1,sentiment_analysis,gpt-5-standard,geographical_bias_100,0.920000,0.930000,0.010000,1.090000,0.790000,100
2,sentiment_analysis,gpt-5-standard,length_bias_100,0.970000,0.940000,-0.030000,-3.090000,0.309000,100
3,sentiment_analysis,gpt-5-standard,typo_bias_100,0.950000,0.960000,0.010000,1.050000,0.735000,100
4,sentiment_analysis,gpt-5-standard,capitalization_100,0.960000,0.960000,-0.000000,-0.000000,1.000000,99
5,sentiment_analysis,gpt-5-standard,punctuation_100,0.910000,0.940000,0.030000,3.300000,0.423000,100
6,sentiment_analysis,gpt-5-standard,derivation_100,0.954000,0.943000,-0.011000,-1.200000,0.734000,87
7,sentiment_analysis,gpt-5-standard,compound_word_100,0.968000,0.958000,-0.011000,-1.090000,0.702000,95
8,sentiment_analysis,gpt-5-standard,active_to_passive_100,0.950000,0.930000,-0.020000,-2.110000,0.554000,100
9,sentiment_analysis,gpt-5-standard,grammatical_role_100,0.955000,0.955000,-0.000000,-0.000000,1.000000,66


## Model Comparison

In [ ]:
# Compare GPT-5 with other models
comparison_files = {
    'GPT-5': f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv',
    'GPT-4o': 'results/sa/gpt4o-0shot-sst2.csv',
    'Claude-3.5': 'results/sa/claude-3-5-sonnet-0shot-sst2.csv',
    'o3-2025-04-16': 'results/sa/o3-2025-04-16-standard-0shot-sst2.csv',
    'Mixtral-8x22B': 'results/sa/mixtral-8x22b-sst2.csv'
}

comparison_df = compare_models(comparison_files, task_name='sentiment_analysis')

if not comparison_df.empty:
    print("\nModel Comparison (including GPT-5):")
    print(comparison_df)
    
    # Calculate GPT-5 improvement
    if 'GPT-5' in comparison_df['Model'].values:
        gpt5_acc = comparison_df[comparison_df['Model'] == 'GPT-5']['Accuracy'].values[0]
        other_accs = comparison_df[comparison_df['Model'] != 'GPT-5']['Accuracy'].values
        if len(other_accs) > 0:
            avg_others = other_accs.mean()
            improvement = gpt5_acc - avg_others
            print(f"\nGPT-5 Performance: {gpt5_acc:.3f}")
            print(f"Average of other models: {avg_others:.3f}")
            print(f"GPT-5 improvement: {improvement:+.3f} ({improvement*100:+.1f}%)")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## GPT-5 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Sentiment Analysis with GPT-5 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nGPT-5 Configuration: {config['description']}")
print(f"\nKey advantages of GPT-5:")
print("• Enhanced reasoning capabilities beyond GPT-4")
print("• Better linguistic robustness")
print("• Faster inference than o3 models")
print("• Higher throughput with multi-threading")

print(f"\nFiles saved in: results/sa/")